In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm

from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf, acf, pacf
from sklearn.metrics import mean_squared_error

In [ ]:
air_passengers = pd.read_csv("https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv")
air_passengers

## Introducción

En este notebook se explorará **cómo se construyen los gráficos ACF y PACF** desde cero, con el objetivo de entender su significado y utilidad al modelar series de tiempo con ARIMA.

Ambos gráficos son herramientas fundamentales para determinar los parámetros del modelo:

- **$p$**: número de términos autorregresivos (AR), sugerido por el gráfico **PACF**
- **$q$**: número de términos de media móvil (MA), sugerido por el gráfico **ACF**

A lo largo del notebook se visualizarán:

- **Nubes de dispersión** entre $y_t$ y sus rezagos $y_{t-k}$
- **Cálculos explícitos** de las correlaciones entre estos pares
- Comparaciones entre ACF y PACF para entender cómo y por qué se diferencian

El objetivo es lograr una comprensión visual e intuitiva de **por qué los gráficos ACF y PACF tienen la forma que tienen** y cómo eso nos ayuda a construir un buen modelo ARIMA.

### ¿Qué es el gráfico ACF?

El gráfico ACF (*Autocorrelation Function*) muestra la correlación entre la serie original y sus propios rezagos.

Para cada rezago $k$, se calcula la correlación entre $y_t$ y $y_{t-k}$.

Este gráfico ayuda a identificar la dependencia temporal en la serie y es clave para estimar el parámetro $q$ (media móvil) en modelos ARIMA.

In [ ]:
air_passengers

In [ ]:
air_passengers['Month'] = pd.to_datetime(air_passengers['Month'])
air_passengers.set_index('Month', inplace=True)
serie = air_passengers['Passengers']

In [ ]:
serie.diff().shift()

In [ ]:
# Crear DataFrame con rezagos
df_lags = pd.DataFrame({
    #'serie': serie,
    'y_t': serie.diff(),
    'y_t-1': serie.diff().shift(2)
}).dropna()

In [ ]:
df_lags

In [ ]:
# Ajustar regresión lineal
x = df_lags['y_t-1']
y = df_lags['y_t']
coef = np.polyfit(x, y, 1)
regresion = np.poly1d(coef)

In [ ]:
regresion

In [ ]:
# Graficar
plt.figure(figsize=(6, 6))
plt.scatter(x, y, alpha=0.6, label='Datos')
plt.plot(x, regresion(x), color='red', label='Regresión lineal')
plt.xlabel('$y_{t-2}$')
plt.ylabel('$y_t$')
plt.title('Relación entre $y_t$ y $y_{t-2}$ (lag 1)')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Extraer variables
x = df_lags['y_t-1']
y = df_lags['y_t']

# Calcular regresión lineal
coef = np.polyfit(x, y, 1)  # coef[0] = pendiente, coef[1] = intercepto

# Mostrar ecuación de la recta
print(f'Regresión lineal: y_t = {coef[0]:.4f} * y_(t-1) + {coef[1]:.4f}')

In [ ]:
# Calcular la correlación de Pearson entre y_t y y_{t-1}
correlacion = df_lags['y_t'].corr(df_lags['y_t-1'])

# Mostrar resultado
print(f'Correlación de Pearson (lag 1): {correlacion:.4f}')

In [ ]:
acf_plot = plot_acf(serie.diff().dropna(), lags = 40)

### ¿Qué es el gráfico PACF?

El gráfico PACF (*Partial Autocorrelation Function*) muestra la **correlación directa** entre la serie $y_t$ y su rezago $y_{t-k}$, **eliminando el efecto de los lags intermedios** como $y_{t-1}, y_{t-2}, \dots, y_{t-(k-1)}$.

A diferencia del ACF, que mide correlaciones acumuladas, el PACF permite identificar el **impacto puro** de cada rezago individual.

Este gráfico es fundamental para estimar el parámetro $p$ (autorregresivo) en modelos ARIMA.


In [ ]:
# Crear DataFrame con los rezagos necesarios
df_pacf = pd.DataFrame({
    'y_t': serie.diff(),
    'y_t-1': serie.diff().shift(1),
    'y_t-2': serie.diff().shift(2)
}).dropna()

In [ ]:
serie.diff()

In [ ]:
df_pacf

In [ ]:
# Obtener residuos de y_t ~ y_{t-1}
res_y = sm.OLS(df_pacf['y_t'], sm.add_constant(df_pacf[['y_t-1']])).fit().resid

# Obtener residuos de y_{t-2} ~ y_{t-1}
res_x = sm.OLS(df_pacf['y_t-2'], sm.add_constant(df_pacf[['y_t-1']])).fit().resid

In [ ]:
pd.DataFrame({
    'resid1': res_y,
    'resid2': res_x
})

In [ ]:
# Ajustar regresión lineal
x = res_x
y = res_y
coef = np.polyfit(x, y, 1)
regresion = np.poly1d(coef)
regresion

In [ ]:
# Graficar residuos para visualizar PACF lag 2
plt.figure(figsize=(6, 6))
plt.scatter(res_x, res_y, alpha=0.6)
plt.xlabel('Residuos de $y_{t-2}$ (quitando $y_{t-1}$)')
plt.ylabel('Residuos de $y_t$ (quitando $y_{t-1}$)')
plt.title('Visualización del PACF en lag 2')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Calcular la correlación entre residuos: esto es el PACF en lag 2
pacf_lag2 = res_y.corr(res_x)

# Mostrar resultado
print(f'PACF (lag 2): {pacf_lag2:.4f}')

In [ ]:
pacf_plot = plot_pacf(serie.diff().dropna(), lags = 40)